# HPAFL: Hybrid Privacy-Aware Federated Learning on Kaggle T4 GPU

This notebook implements a complete Hybrid Privacy-Aware Federated Learning (HPAFL) system on the Kaggle T4 GPU. It demonstrates:
- **Federated Learning** with multiple hospital clients
- **Differential Privacy** using Opacus DP-SGD
- **Secure Aggregation** with pairwise masking
- **Adaptive Weighted Aggregation** based on accuracy, data quality, and reliability
- **FedBN** (Batch Normalization not aggregated across hospitals)

**Dataset**: HAM10000 (10,000 skin lesion images, 7 classes)

**GPU**: T4 (16GB VRAM - sufficient for batch_size 64 with DP-SGD)

**Runtime**: ~20 minutes for 2 rounds with 1 epoch per hospital

## Cell 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision efficientnet_pytorch opacus scikit-learn pandas numpy matplotlib tqdm

## Cell 2: Setup Environment and Verify GPU

In [ ]:
import os
import sys
from pathlib import Path

# Setup paths
os.chdir('/kaggle/working')
sys.path.insert(0, '/kaggle/working')

# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
    print("✓ GPU ready for training!")
else:
    print("✗ GPU not available. CPU training will be very slow.")

## Cell 3: Clone HPAFL Framework

In [ ]:
import subprocess

# Clone the framework (replace with your actual repo URL)
repo_url = "https://github.com/YOUR-USERNAME/hpafl-framework.git"

# Check if already cloned
if not Path('/kaggle/working/hpafl-framework').exists():
    print(f"Cloning from {repo_url}...")
    result = subprocess.run(
        ['git', 'clone', repo_url],
        cwd='/kaggle/working',
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ Framework cloned successfully")
    else:
        print(f"✗ Clone failed. Please upload files manually.")
        print(f"Error: {result.stderr}")
else:
    print("✓ Framework already exists")

os.chdir('/kaggle/working/hpafl-framework')
sys.path.insert(0, '/kaggle/working/hpafl-framework')
print(f"Working directory: {os.getcwd()}")

## Cell 4: Import HPAFL Modules

In [ ]:
import logging
import json
from pathlib import Path

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
logger = logging.getLogger(__name__)

# Import HPAFL modules
from config import HAPFLConfig
from scripts.run_hpafl import run_hpafl

print("✓ HPAFL modules imported successfully")

## Cell 5: Configure HPAFL for Kaggle

In [ ]:
# Create configuration
cfg = HAPFLConfig()

# ============ Dataset paths (Kaggle-specific) ============
cfg.data_root = "/kaggle/input/skin-cancer-mnist-ham10000"
cfg.output_root = "/kaggle/working"

# ============ Training parameters ============
# Note: These are optimized for T4 GPU (16GB VRAM)
cfg.num_rounds = 2           # Reduced for demo (use 20 for full training)
cfg.local_epochs = 1         # Reduced for demo (use 5 for full training)
cfg.batch_size = 64          # Safe for T4 with DP-SGD + GPU cache clearing
cfg.num_workers = 4          # Kaggle T4 has 4 CPU cores

# ============ Model parameters ============
cfg.model_name = "efficientnet_b0"
cfg.pretrained = True        # Use ImageNet pretrained weights
cfg.image_size = 224

# ============ Privacy parameters (Differential Privacy) ============
cfg.target_epsilon = 8.0     # Privacy budget (8 = moderate privacy)
cfg.target_delta = 1e-5      # Failure probability
cfg.max_grad_norm = 1.0      # Gradient clipping threshold
cfg.noise_multiplier = 1.1   # Gaussian noise scale

# ============ Federated parameters ============
cfg.num_hospitals = 3        # Number of hospital clients
cfg.fraction_fit = 1.0       # All hospitals participate
cfg.fraction_evaluate = 1.0  # Evaluate on all hospitals

# ============ Adaptive aggregation weights (must sum to 1.0) ============
cfg.alpha_accuracy = 0.40    # Weight on accuracy
cfg.alpha_reliability = 0.30 # Weight on reliability
cfg.alpha_data_quality = 0.20 # Weight on data quality
cfg.alpha_historical = 0.10  # Weight on historical performance
cfg.ema_decay = 0.30         # Exponential moving average decay

print("="*70)
print("HPAFL Configuration for Kaggle T4 GPU")
print("="*70)
print(f"Data root:              {cfg.data_root}")
print(f"Output root:            {cfg.output_root}")
print(f"\nTraining Settings:")
print(f"  Num rounds:           {cfg.num_rounds}")
print(f"  Local epochs:         {cfg.local_epochs}")
print(f"  Batch size:           {cfg.batch_size}")
print(f"  Num workers:          {cfg.num_workers}")
print(f"\nModel:                  {cfg.model_name}")
print(f"Image size:             {cfg.image_size}x{cfg.image_size}")
print(f"Pretrained:             {cfg.pretrained}")
print(f"\nDifferential Privacy:")
print(f"  Target ε (epsilon):   {cfg.target_epsilon}")
print(f"  Target δ (delta):     {cfg.target_delta}")
print(f"  Max grad norm:        {cfg.max_grad_norm}")
print(f"  Noise multiplier:     {cfg.noise_multiplier}")
print(f"\nFederated Learning:")
print(f"  Num hospitals:        {cfg.num_hospitals}")
print(f"  Fraction fit:         {cfg.fraction_fit}")
print(f"  Fraction evaluate:    {cfg.fraction_evaluate}")
print("="*70)

## Cell 6: Run HPAFL Training

In [ ]:
# Run the full HPAFL pipeline
try:
    print("\n" + "="*70)
    print("Starting HPAFL Federated Learning Training")
    print("="*70 + "\n")
    
    run_hpafl(cfg)
    
    print("\n" + "="*70)
    print("✓ HPAFL TRAINING COMPLETED SUCCESSFULLY!")
    print("="*70)
    
except Exception as e:
    print(f"\n✗ Error during training: {e}")
    import traceback
    traceback.print_exc()

## Cell 7: Load and Display Results

In [ ]:
import pandas as pd
import json

results_dir = Path(cfg.output_root) / "results"

print("\n" + "="*70)
print("HPAFL RESULTS SUMMARY")
print("="*70)

# 1. Privacy Budget
privacy_file = results_dir / "privacy_budget.csv"
if privacy_file.exists():
    privacy_df = pd.read_csv(privacy_file)
    print("\n📊 Privacy Budget Consumption:")
    print(privacy_df.to_string(index=False))
    print(f"\n  Final ε (epsilon): {privacy_df['epsilon'].iloc[-1]:.4f}")
    print(f"  Target ε: {cfg.target_epsilon:.4f}")
else:
    print("\n⚠️  Privacy budget file not found")

# 2. Adaptive Aggregation Weights
weights_file = results_dir / "adaptive_weights.csv"
if weights_file.exists():
    weights_df = pd.read_csv(weights_file)
    print("\n⚖️  Adaptive Aggregation Weights (by round):")
    for round_num in sorted(weights_df['round'].unique()):
        round_data = weights_df[weights_df['round'] == round_num]
        print(f"\n  Round {round_num}:")
        for _, row in round_data.iterrows():
            print(f"    Hospital {row['hospital_id']}: weight={row['weight']:.4f}, score={row['score']:.4f}")
else:
    print("\n⚠️  Adaptive weights file not found")

# 3. Final Results
results_file = results_dir / "hpafl.json"
if results_file.exists():
    with open(results_file) as f:
        final_results = json.load(f)
    
    print("\n🎯 Final Performance Metrics:")
    print(f"  Global Accuracy:      {final_results.get('global_accuracy', 'N/A'):.4f}")
    print(f"  Global F1 (Macro):    {final_results.get('global_f1_macro', 'N/A'):.4f}")
    print(f"  Total Rounds:         {final_results.get('num_rounds', 'N/A')}")
    print(f"  Final ε (epsilon):    {final_results.get('final_epsilon', 'N/A'):.4f}")
    print(f"  Training Time:        {final_results.get('training_time_minutes', 'N/A'):.2f} minutes")
    
    # Per-hospital results
    if 'hospitals' in final_results:
        print("\n  Per-Hospital Metrics:")
        for hospital_id, metrics in final_results['hospitals'].items():
            print(f"\n    Hospital {hospital_id}:")
            print(f"      Accuracy:  {metrics.get('accuracy', 'N/A'):.4f}")
            print(f"      F1 Macro:  {metrics.get('f1_macro', 'N/A'):.4f}")
            print(f"      Epsilon:   {metrics.get('epsilon', 'N/A'):.4f}")
else:
    print("\n⚠️  Final results file not found")

print(f"\n\n📁 All results saved to: {results_dir}/")
print("="*70)

## Cell 8: Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

results_dir = Path(cfg.output_root) / "results"

# 1. Privacy Budget Over Rounds
privacy_file = results_dir / "privacy_budget.csv"
if privacy_file.exists():
    privacy_df = pd.read_csv(privacy_file)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Privacy budget consumption
    for hospital in privacy_df['hospital_id'].unique():
        hospital_data = privacy_df[privacy_df['hospital_id'] == hospital]
        ax1.plot(hospital_data['round'], hospital_data['epsilon'], marker='o', label=f'Hospital {hospital}')
    
    ax1.axhline(y=cfg.target_epsilon, color='r', linestyle='--', label=f'Target ε={cfg.target_epsilon}')
    ax1.set_xlabel('Round')
    ax1.set_ylabel('ε (Epsilon)')
    ax1.set_title('Privacy Budget Consumption Over Rounds')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Budget remaining
    for hospital in privacy_df['hospital_id'].unique():
        hospital_data = privacy_df[privacy_df['hospital_id'] == hospital]
        ax2.plot(hospital_data['round'], hospital_data['budget_remaining'], marker='s', label=f'Hospital {hospital}')
    
    ax2.set_xlabel('Round')
    ax2.set_ylabel('Budget Remaining')
    ax2.set_title('Privacy Budget Remaining Over Rounds')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(results_dir / 'privacy_budget_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Privacy budget visualization saved")

# 2. Adaptive Weights Over Rounds
weights_file = results_dir / "adaptive_weights.csv"
if weights_file.exists():
    weights_df = pd.read_csv(weights_file)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Weights over rounds
    for hospital in weights_df['hospital_id'].unique():
        hospital_data = weights_df[weights_df['hospital_id'] == hospital]
        ax1.plot(hospital_data['round'], hospital_data['weight'], marker='o', label=f'Hospital {hospital}')
    
    ax1.set_xlabel('Round')
    ax1.set_ylabel('Aggregation Weight')
    ax1.set_title('Adaptive Aggregation Weights Over Rounds')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Composite scores
    for hospital in weights_df['hospital_id'].unique():
        hospital_data = weights_df[weights_df['hospital_id'] == hospital]
        ax2.plot(hospital_data['round'], hospital_data['score'], marker='s', label=f'Hospital {hospital}')
    
    ax2.set_xlabel('Round')
    ax2.set_ylabel('Composite Score')
    ax2.set_title('Hospital Composite Scores Over Rounds')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(results_dir / 'adaptive_weights_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Adaptive weights visualization saved")

## Cell 9: Test Batch Sizes (Optional Diagnostic)

## Cell 10: Export Results for Analysis

In [ ]:
import shutil
from pathlib import Path

results_dir = Path(cfg.output_root) / "results"

# Create a summary file
summary_file = results_dir / "KAGGLE_TRAINING_SUMMARY.txt"

with open(summary_file, 'w') as f:
    f.write("="*70 + "\n")
    f.write("HPAFL Training Summary - Kaggle T4 GPU\n")
    f.write("="*70 + "\n\n")
    
    f.write("CONFIGURATION\n")
    f.write("-" * 70 + "\n")
    f.write(f"Rounds: {cfg.num_rounds}\n")
    f.write(f"Local Epochs: {cfg.local_epochs}\n")
    f.write(f"Batch Size: {cfg.batch_size}\n")
    f.write(f"Model: {cfg.model_name}\n")
    f.write(f"Image Size: {cfg.image_size}\n")
    f.write(f"Num Hospitals: {cfg.num_hospitals}\n\n")
    
    f.write("PRIVACY PARAMETERS\n")
    f.write("-" * 70 + "\n")
    f.write(f"Target Epsilon: {cfg.target_epsilon}\n")
    f.write(f"Target Delta: {cfg.target_delta}\n")
    f.write(f"Max Grad Norm: {cfg.max_grad_norm}\n")
    f.write(f"Noise Multiplier: {cfg.noise_multiplier}\n\n")
    
    f.write("OUTPUT FILES\n")
    f.write("-" * 70 + "\n")
    for file in sorted(results_dir.glob('*')):
        if file.is_file():
            size_mb = file.stat().st_size / 1e6
            f.write(f"  {file.name} ({size_mb:.2f} MB)\n")

print(f"\n✓ Summary saved to {summary_file}")

# List all output files
print("\n" + "="*70)
print("OUTPUT FILES")
print("="*70)
for file in sorted(results_dir.glob('*')):
    if file.is_file():
        size_mb = file.stat().st_size / 1e6
        print(f"  {file.name:<40} {size_mb:>8.2f} MB")